# Jev classification benchmark: Colab GPU runner

Run the same package used locally. Select **Runtime → Change runtime type → GPU**. This notebook makes **no hosted model API calls**. Colab compute availability, quotas, and any compute charges depend on your account; a GPU session is not guaranteed.

For this private repository, create a source bundle locally with `python scripts/export_colab_bundle.py`, then upload the resulting `artifacts/jev-classification-benchmark-colab.zip`. The exporter allowlists source/configuration files and removes saved notebook outputs. Review source cells for manually pasted secrets before sharing. It excludes datasets, models, `.env` files, and measured results. Use Colab Secrets for credentials; never paste credentials into a saved notebook.

The next cell verifies the source manifest and extracts into a **fresh temporary directory**, preserving existing sessions. Pinned source data and fixed preparation settings reconstruct the same split; compare test-manifest hashes against earlier results before treating them as paired experiments. Archive the prepared data privately if needed for exact recovery.

If the upload picker in the next cell does not open, use Colab's **Files pane → Upload to session storage** to upload `jev-classification-benchmark-colab.zip`, then run the cell. It automatically uses that file from `/content`; all archive and checksum checks still run.


In [ ]:
import hashlib, io, json, stat, sys, tempfile, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files

assert (3, 11) <= sys.version_info[:2] < (3, 14), 'Use Python 3.11–3.13'
bundle_path = Path('/content/jev-classification-benchmark-colab.zip')
uploaded = {bundle_path.name: bundle_path.read_bytes()} if bundle_path.is_file() else files.upload()
zips = [name for name in uploaded if name.endswith('.zip')]
if len(zips) != 1:
    raise ValueError('Upload exactly one exported source ZIP')
bundle_root = 'jev-classification-benchmark'
with zipfile.ZipFile(io.BytesIO(uploaded[zips[0]])) as archive:
    members = archive.infolist()
    names = [member.filename for member in members]
    if len(names) != len(set(names)) or sum(m.file_size for m in members) > 25_000_000:
        raise ValueError('Duplicate names or unexpectedly large source bundle')
    for member in members:
        path = PurePosixPath(member.filename)
        mode = member.external_attr >> 16
        if path.is_absolute() or '..' in path.parts or path.parts[0] != bundle_root or stat.S_ISLNK(mode):
            raise ValueError('Unsafe source archive member')
    manifest = json.loads(archive.read(f'{bundle_root}/bundle_manifest.json'))
    expected = manifest['files_sha256']
    if set(names) != {f'{bundle_root}/{name}' for name in expected} | {f'{bundle_root}/bundle_manifest.json'}:
        raise ValueError('Source bundle manifest/file mismatch; re-export the source ZIP')
    for name, digest in expected.items():
        if hashlib.sha256(archive.read(f'{bundle_root}/{name}')).hexdigest() != digest:
            raise ValueError(f'Source checksum mismatch: {name}')
    workspace = Path(tempfile.mkdtemp(prefix='jevbench-', dir='/content'))
    archive.extractall(workspace)
project = workspace / bundle_root
del uploaded
%cd {project}
%pip install -q -e '.[dev,neural,qlora]'


In [ ]:
# Colab only: its optional preinstalled torchao can conflict with PEFT adapter loading.
# This benchmark uses bitsandbytes for QLoRA and does not require torchao.
# Run before package/GPU capture so exported metadata records the corrected environment.
%pip uninstall -y torchao


## Fixed pilot and execution budget

The default workflow runs **SST-2 and TREC**, each with four classical baselines and three local model conditions: zero-shot, four-shot-per-class and QLoRA. `K` is examples **per class**, shared by prompting, QLoRA and classical ML: eight new labels for SST-2 and 24 for TREC. The core settings are split seed 42, selection/training seed 42, at most 10,000 training / 1,000 development / 200 test rows **per dataset**, and the same 2,000-character prefix for every method. These are pilot results, not full official-test scores. Use seeds 13/42/87 in a separately declared larger study without changing the heldout preparation seed.

`qwen_main` is the 4B main candidate. `qwen_small` is the 0.5B technical smoke baseline, useful to check the pipeline first. Each local model evaluation is capped at 200 decisions: six neural evaluations across both datasets, plus eight classical runs. This does not bound GPU hours. Qwen 4B full-weight evaluation can still exhaust memory even though training uses QLoRA. No automatic downgrade, example removal or extra model purchase is performed. Keep the model and training recipe fixed for both datasets before examining test outcomes.


In [ ]:
import os, subprocess
os.environ['HF_HOME'] = str((project / 'data/hf').resolve())
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running these cells'
print('GPU:', torch.cuda.get_device_name(0))
import importlib.metadata, platform
environment_record = {
    'python': platform.python_version(), 'platform': platform.platform(),
    'gpu': torch.cuda.get_device_name(0),
    'gpu_memory_bytes': torch.cuda.get_device_properties(0).total_memory,
    'torch_cuda_version': torch.version.cuda,
    'packages': dict(sorted((d.metadata['Name'], d.version) for d in importlib.metadata.distributions() if d.metadata.get('Name'))),
}
(project / 'results').mkdir(exist_ok=True)
(project / 'results/colab_environment.json').write_text(json.dumps(environment_record, indent=2))

def run(*args):
    command = [sys.executable, '-u', '-m', 'jevbench.cli', *map(str, args)]
    with subprocess.Popen(command, cwd=project, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            print(line, end='', flush=True)
        if proc.wait():
            raise RuntimeError('Benchmark command failed; inspect output above')

DATASET = 'sst2'  # first dataset; the next section runs TREC separately
K = 4
SEED = 42
MODEL_KEY = 'qwen_main'  # choose qwen_small for a technical smoke run
MAX_TEST_REQUESTS = 200
models = json.loads((project / 'configs/models.json').read_text())
model = models[MODEL_KEY]
assert model['provider'] in {'hf', 'huggingface'}, 'This notebook permits local weights only'
assert len(model['revision']) == 40, 'Pin a complete model commit'
adapter = Path('models') / f"{model['model'].split('/')[-1]}-{DATASET}-k{K}-s{SEED}"

run('prepare', DATASET, '--output', 'data/pilot', '--seed', 42,
    '--train-limit', 10000, '--validation-limit', 1000, '--test-limit', MAX_TEST_REQUESTS,
    '--max-text-chars', 2000)
run('classical', '--data', f'data/pilot/{DATASET}', '--train-per-class', K,
    '--seed', SEED, '--output', 'results/colab')


## Base model: zero-shot and few-shot

Local scoring uses each complete numeric label plus EOS, normalized across classes. It is not free-form generation. Full selected prompts must fit the configured context; over-context inputs fail instead of silently dropping examples. Banking77 requires 77 candidate forward passes per row and its few-shot prompts may not fit, so it is not a suitable first smoke run.


In [ ]:
for shots in (0, K):
    run('model', '--data', f'data/pilot/{DATASET}', '--config', 'configs/models.json',
        '--model-key', MODEL_KEY, '--shots', shots, '--seed', SEED,
        '--output', 'results/colab', '--max-requests', MAX_TEST_REQUESTS)


## QLoRA: the same K × classes training labels

Train one fixed final checkpoint for 3 epochs, rank 8/alpha 16/all-linear targets, without using development labels for selection. Training uses 4-bit NF4 with double quantization; the backend selects and records its compute dtype. Evaluation reloads base weights at the configured normal inference precision with the adapter. Base and adapted evaluation share the same scoring/precision, but adapter training is quantized: label the method **QLoRA**, not full-precision LoRA.

A completed adapter in this session is reused; an incomplete nonempty directory is never overwritten. Changing the training recipe requires a new output directory. The CLI validates saved training provenance before evaluation. Skipping this cell still permits export of earlier baseline results.


In [ ]:
metadata_path = project / adapter / 'jevbench_training.json'
if not metadata_path.exists():
    run('train-lora', '--data', f'data/pilot/{DATASET}', '--model', model['model'],
        '--revision', model['revision'], '--output', adapter, '--train-per-class', K,
        '--seed', SEED, '--epochs', 3, '--max-length', 2048, '--load-in-4bit')
else:
    saved = json.loads(metadata_path.read_text())
    assert (saved['model_id'], saved['dataset'], saved['seed'], saved['train_per_class'], saved['epochs']) == (model['model'], DATASET, SEED, K, 3)
    assert saved['resolved_revision'] == model['revision'] and saved['load_in_4bit']
    print('Reusing existing adapter; no new training')
models['colab_adapter'] = {**model, 'adapter_path': str(adapter)}
(project / 'configs/colab_models.json').write_text(json.dumps(models, indent=2))
run('model', '--data', f'data/pilot/{DATASET}', '--config', 'configs/colab_models.json',
    '--model-key', 'colab_adapter', '--seed', SEED, '--output', 'results/colab',
    '--max-requests', MAX_TEST_REQUESTS)


## TREC: the same fixed protocol on six classes

Run the second dataset with the same model, selection seed, input policy and four examples per class. This adds four matched-budget classical runs and zero-shot, few-shot and QLoRA model runs, each on the same 200 prepared TREC test rows. The TREC adapter uses 24 training labels and its own directory; it does not reuse SST-2 weights. The three-epoch recipe and final-checkpoint rule remain fixed regardless of the first dataset's test scores. No development labels are used to select either adapter.


In [ ]:
TREC_DATASET = 'trec'
trec_adapter = Path('models') / f"{model['model'].split('/')[-1]}-{TREC_DATASET}-k{K}-s{SEED}"
run('prepare', TREC_DATASET, '--output', 'data/pilot', '--seed', 42,
    '--train-limit', 10000, '--validation-limit', 1000, '--test-limit', MAX_TEST_REQUESTS,
    '--max-text-chars', 2000)
run('classical', '--data', f'data/pilot/{TREC_DATASET}', '--train-per-class', K,
    '--seed', SEED, '--output', 'results/colab')
for shots in (0, K):
    run('model', '--data', f'data/pilot/{TREC_DATASET}', '--config', 'configs/models.json',
        '--model-key', MODEL_KEY, '--shots', shots, '--seed', SEED,
        '--output', 'results/colab', '--max-requests', MAX_TEST_REQUESTS)

trec_metadata_path = project / trec_adapter / 'jevbench_training.json'
if not trec_metadata_path.exists():
    run('train-lora', '--data', f'data/pilot/{TREC_DATASET}', '--model', model['model'],
        '--revision', model['revision'], '--output', trec_adapter, '--train-per-class', K,
        '--seed', SEED, '--epochs', 3, '--max-length', 2048, '--load-in-4bit')
else:
    saved = json.loads(trec_metadata_path.read_text())
    assert (saved['model_id'], saved['dataset'], saved['seed'], saved['train_per_class'], saved['epochs']) == (model['model'], TREC_DATASET, SEED, K, 3)
    assert saved['resolved_revision'] == model['revision'] and saved['load_in_4bit']
    assert (saved['r'], saved['lora_alpha'], saved['learning_rate'], saved['max_length']) == (8, 16, 0.0002, 2048)
    assert saved['validation_rows'] == 0 and not saved['test_accessed']
    print('Reusing existing TREC adapter; no new training')
models['colab_adapter'] = {**model, 'adapter_path': str(trec_adapter)}
(project / 'configs/colab_models.json').write_text(json.dumps(models, indent=2))
run('model', '--data', f'data/pilot/{TREC_DATASET}', '--config', 'configs/colab_models.json',
    '--model-key', 'colab_adapter', '--seed', SEED, '--output', 'results/colab',
    '--max-requests', MAX_TEST_REQUESTS)


## Save measured results

Download results before the runtime expires. The exporter below includes only known run records, label-only prediction files, text-free test manifests, reports, package/GPU metadata, and metadata for every completed adapter (SST-2 and TREC). Optional before/after environment-repair records are retained when present. It excludes raw corpus text, notebook outputs, environment secrets, unrelated results, and weights. Review artifacts before making them public. Save source bundle and package versions with the experiment.

To reuse trained adapters after Colab resets, separately copy the adapter directory and its metadata to private storage; review model/dataset licenses before publishing weights. Hosted Jev/frontier runs need separately configured secrets and a total spending ceiling. This notebook does not authorize or execute those calls. A request-count cap is not a dollar ceiling.


In [ ]:
run('report', '--results', 'results/colab', '--output', 'results/COLAB_REPORT.md')
export_files = []
for name in ('run.json', 'predictions.jsonl', 'test_manifest.json'):
    export_files.extend((project / 'results/colab').glob(f'*/{name}'))
for name in ('COLAB_REPORT.md', 'COLAB_REPORT.csv', 'colab_environment.json',
             'colab_environment_before_fix.json', 'colab_environment_after_fix.json'):
    path = project / 'results' / name
    if path.is_file():
        export_files.append(path)
archive_path = workspace / 'jevbench-colab-results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(set(export_files)):
        archive.write(path, path.relative_to(project).as_posix())
    # Include all completed adapters' metadata, never adapter/base weight files.
    for metadata_path in sorted((project / 'models').glob('*/jevbench_training.json')):
        archive.write(metadata_path, f'results/adapter_metadata/{metadata_path.parent.name}.json')
files.download(str(archive_path))
